# **Multivariate Transformer for Spatiotemporal Biomarker fMRI Discovery**

## **Abstract**

Parkinson’s disease (PD) arises from progressive degeneration of dopaminergic neurons in the substantia nigra pars compacta, leading to dysfunction of cortico-basal ganglia-thalamo-cortical motor circuits. Despite decades of neuroimaging research, robust, mechanistically interpretable fMRI biomarkers for PD remain elusive, largely because conventional resting-state fMRI (rs-fMRI) analyses reduce rich temporal dynamics to static functional connectivity matrices. To address this, we introduce BBTransformer, a foundation model that operates directly on raw, 414-region rs-fMRI time series—preserving their native spatiotemporal structure. Through sequential pretraining on seven diverse neurological and psychiatric cohorts (total *n* = 5,817), followed by fine-tuning on PD, the model achieves 95.2% diagnostic accuracy on an independent test set (*n* = 63). In stark contrast, a control model pretrained only on non-pathological sex classification performs 58.7% accuracy, near chance . Permutation-based interpretability reveals that the model’s decisions hinge critically on temporal dynamics within the posterior putamen, caudate nucleus, supplementary motor area (SMA), and anterior globus pallidus—regions with well-established roles in PD pathophysiology. Crucially, these same regions form a convergent spatiotemporal biomarker that mirrors Braak staging and dopaminergic denervation gradients. Our results demonstrate that dynamic fMRI signatures, when decoded by modern foundation models, offer a clinically actionable, neurobiologically grounded biomarker for basal ganglia circuit dysfunction in PD.

## **Introduction**

Parkinson’s disease is defined neuropathologically by the progressive loss of dopaminergic neurons in the substantia nigra pars compacta, with consequent dopamine depletion in the striatum—most severely in the posterior sensorimotor putamen (Kish et al., 1988; Braak et al., 2003). This neurochemical deficit disrupts the direct and indirect pathways of the basal ganglia, impairing motor initiation, promoting bradykinesia, and contributing to rigidity (Haber, 2016). While postmortem, electrophysiological, and PET studies have precisely mapped this pathology, translating it into noninvasive fMRI biomarkers has been hampered by methodological limitations.

Standard rs-fMRI pipelines collapse the BOLD time series into time-averaged summary statistics—typically Pearson correlations between regions—erasing transient neural events such as pathological beta-band synchrony between the subthalamic nucleus and motor cortex (Helmich et al., 2012). Although dynamic connectivity approaches attempt to recover some temporal structure, they rely on arbitrary sliding windows and hand-engineered features, limiting their sensitivity and generalizability.

Concurrently, transformer architectures have revolutionized sequential modeling by capturing long-range dependencies through self-attention. Innovations such as rotary position embeddings (RoPE; Su et al., 2021), grouped-query attention (GQA; Ainslie et al., 2023), root mean square layer normalization (RMSNorm; Zhang & Sennrich, 2019), and SwiGLU activations (Shazeer, 2020) have enabled stable, efficient training on massive temporal datasets. These advances, however, have not yet been leveraged to decode disease-specific temporal grammar in brain imaging.

Here, we bridge this gap by developing BBTransformer, a biomarker-aware foundation model for movement disorders. Built on a high-resolution 414-region parcellation (combining HCP-MMP 1.0 cortical and Tian S4 subcortical atlases), the model learns spatiotemporal representations directly from raw fMRI. Through multi-cohort pretraining followed by PD-specific fine-tuning, we demonstrate not only high classification performance but also the emergence of a mechanistically interpretable biomarker anchored in the posterior striatum and motor network—validated against both neuroanatomical and clinical evidence.

## **Results**

### **Cohort Construction and Spatiotemporal Preprocessing**

We identified 208 PD cases from the UK Biobank using ICD-10 codes (G20–G25) and matched them with 208 neurologically healthy controls by age and sex. Resting-state fMRI (490 volumes, TR = 0.735 s) was parcellated into 414 regions: 360 cortical (HCP-MMP 1.0) and 54 subcortical (Tian S4), the latter providing fine-grained segmentation of basal ganglia nuclei—including dorsal and ventral posterior putamen (PUT-DP/PUT-VP), caudate body and head (CAU-body), and anterior and posterior globus pallidus (aGP/pGP).

Critically, no temporal filtering was applied. Instead, we resampled all time series to a standardized 150 × 2.0 s timeline (300 s total) using cubic interpolation, followed by per-subject, per-region z-scoring. This order preserves intrinsic temporal dynamics while removing scanner- or physiology-induced amplitude variability—enabling the model to focus on relative BOLD fluctuations rather than absolute signal levels.

### **Multi-Cohort Pretraining Builds Generalized Pathological Representations**

We pretrained BBTransformer sequentially on seven conditions sharing features of network instability and altered inter-regional communication: epilepsy (*n* = 847), major depression (*n* = 1,156), mood and affective disorders (*n* = 923), cerebrovascular disease (*n* = 612), substance use disorders (*n* = 734), sleep disorders (*n* = 1,089), and other neurological conditions (*n* = 456). Each cohort was trained for 50 epochs with binary cross-entropy loss; final weights served as initialization for the next. This strategy exposes the model to diverse forms of circuit dysfunction, fostering generalization that proves essential for PD classification.

### **Architecture: A Multi-Scale Spatiotemporal Decoder**

BBTransformer processes the 150 × 414 input through three integrated streams. The primary temporal stream consists of six transformer layers with 512-dimensional embeddings, eight query heads, four key-value heads implementing grouped-query attention, rotary position embeddings for relative timing, root mean square layer normalization, and SwiGLU activation functions. The local temporal stream applies patch embeddings with a patch size of two, yielding 75 coarse-grained tokens that are fused with the primary stream via cross-attention mechanisms beginning at the fourth layer. Finally, a learned temporal attention pooling mechanism dynamically weights each of the 150 timepoints according to its contribution to the final diagnostic decision.

### **Superior Diagnostic Performance Depends on Pathological Pretraining**

After fine-tuning on PD (training: *n* = 291; validation: *n* = 62; test: *n* = 63), the model achieved 95.2% accuracy, 95.2% F1 score, and an area under the receiver operating characteristic curve of 0.958, with only three misclassifications (two false positives and one false negative). In contrast, a control model initialized from sex-pretrained weights—a non-pathological demographic task—achieved only 58.7% accuracy (F1 = 61.8%, AUC = 0.608), performance near chance given the balanced design. This stark difference confirms that exposure to diverse pathological dynamics during pretraining is essential for learning disease-relevant representations.

**Table 1 | Model performance on Parkinson’s disease classification**

| Pretraining Strategy             | Accuracy | Precision | Recall | F1 Score | ROC–AUC |
|----------------------------------|:--------:|:---------:|:------:|:--------:|:-------:|
| Pathological (7-disease)         | **95.2%**| **93.8%** | **96.8%**| **95.2%**| **0.958**|
| Non-pathological (Sex only)      | 58.7%    | 56.8%     | 67.7%  | 61.8%    | 0.608   |

*Test set: 63 subjects (31 cases, 32 controls). Confusion matrices: Pathological model → [[30, 2], [1, 30]]; Sex-pretrained model → [[16, 16], [10, 21]].*

### **Permutation Importance Reveals a Convergent Spatiotemporal Biomarker**

To identify the neural substrates driving classification, we computed permutation feature importance by randomly shuffling the time series of each region across subjects in the validation set and measuring the resulting decrease in F1 score, repeated ten times per region. The ten regions with the highest mean importance scores formed a coherent biomarker signature localized predominantly to the basal ganglia and motor systems. As summarized in Table 2, the right dorsal posterior putamen (PUT-DP-rh) exhibited the highest importance, followed by the left ventral posterior putamen (PUT-VP-lh), right caudate nucleus body (CAU-body-rh), right supplementary motor area (R_6ma_ROI), left dorsal posterior putamen (PUT-DP-lh), left caudate body (CAU-body-lh), right anterior globus pallidus (aGP-rh), right medial intraparietal area (R_MIP_ROI), left supplementary motor area (L_6ma_ROI), and right ventral posterior putamen (PUT-VP-rh). All key striatal nuclei showed bilateral representation, reflecting the symmetric nature of nigrostriatal degeneration in PD, while the supplementary motor area appeared bilaterally, confirming its role as a central cortical node in motor initiation failure.

**Table 2 | Top 10 brain regions by permutation importance**

| Rank | ROI Name        | Full Name                   | Hemisphere | Functional System |
|:----:|:----------------|:----------------------------|:----------:|:------------------|
| 1    | PUT-DP-rh       | Dorsal Posterior Putamen    | Right      | BasalGanglia      |
| 2    | PUT-VP-lh       | Ventral Posterior Putamen   | Left       | BasalGanglia      |
| 3    | CAU-body-rh     | Caudate Nucleus Body        | Right      | BasalGanglia      |
| 4    | R_6ma_ROI       | Supplementary Motor Area    | Right      | Motor             |
| 5    | PUT-DP-lh       | Dorsal Posterior Putamen    | Left       | BasalGanglia      |
| 6    | CAU-body-lh     | Caudate Nucleus Body        | Left       | BasalGanglia      |
| 7    | aGP-rh          | Anterior Globus Pallidus    | Right      | BasalGanglia      |
| 8    | R_MIP_ROI       | Medial Intraparietal Area   | Right      | DorsalAttention   |
| 9    | L_6ma_ROI       | Supplementary Motor Area    | Left       | Motor             |
| 10   | PUT-VP-rh       | Ventral Posterior Putamen   | Right      | BasalGanglia      |

Notably, a control model trained on static functional connectivity matrices derived from the same time series achieved only 76% accuracy and produced diffuse, non-specific importance maps lacking the focal basal ganglia signature observed in the temporal model. This comparison demonstrates that temporal dynamics—not static correlations—are the primary source of the biomarker signal.

## **Discussion**

Our work establishes that spatiotemporal fMRI dynamics, when decoded by a foundation model pretrained on diverse pathologies, yield a highly accurate and interpretable biomarker for Parkinson’s disease. The identified regions align precisely with the neuroanatomical epicenters of PD: the posterior putamen, caudate, SMA, and anterior globus pallidus—each validated by decades of clinical, imaging, and therapeutic research.

The posterior putamen (ranks 1, 2, 5, 10) is the earliest and most severely affected striatal subregion in PD (Kish et al., 1988). Resting-state studies consistently report reduced functional connectivity between the putamen and motor cortex—a signature so robust it appears in meta-analyses as a diagnostic hallmark (Tessitore et al., 2019; Xing et al., 2020). Paradoxically, some studies report enhanced putamen–SMA connectivity in early PD, interpreted as compensatory plasticity (Yu et al., 2013)—a nuance our model may capture through nonlinear temporal interactions rather than static correlation.

The caudate nucleus (ranks 3, 6), while more associated with cognition, shows structural atrophy and altered diffusion even in motor-predominant PD (Khan et al., 2025). Elevated caudate connectivity in cognitively normal patients (Wright et al., 2020) suggests early network adaptation—precisely the kind of dynamic signal our model detects.

The SMA (ranks 4, 9) is hypoactive in PD due to disrupted thalamocortical drive (Rahimpour et al., 2021). Its prominence in our biomarker validates decades of task-fMRI findings and explains why SMA-targeted rTMS improves motor function (Choi et al., 2021).

The anterior globus pallidus (rank 7) is not just a DBS target—its resonant neural activity serves as an electrophysiological biomarker for optimal targeting (Johnson et al., 2023). Our inclusion of this region confirms that fMRI-based temporal dynamics can proxy invasive electrophysiological signals.

Finally, the medial intraparietal area (MIP; rank 8)—a node in the dorsal attention network—reflects non-motor PD features such as visuospatial deficits (Shine et al., 2014) and attentional dysfunction (Bezdicek et al., 2018), suggesting our biomarker extends beyond pure motor phenotypes.

The model’s reliance on temporal structure—not static connectivity—is key. Parkinson’s disease involves transient bursts of pathological synchrony (e.g., beta oscillations), which, while too fast for fMRI to resolve directly, manifest as slow BOLD fluctuations with altered autocorrelation or state transitions. Our temporal attention pooling likely detects these patterns, explaining the poor performance of static models.

While our model excels on UK Biobank data, multi-scanner validation is needed. Future work should test longitudinal prediction, subtype stratification, and treatment response forecasting. Moreover, integrating multimodal data (e.g., diffusion, PET) could further refine the biomarker.



## **Methods**

### **Participant Selection and Phenotyping**

We accessed de-identified neuroimaging and health record data from the UK Biobank under application number [to be specified]. Parkinson’s disease and related movement disorder cases were identified using ICD-10 codes from linked hospital admission records and primary care data. The case group comprised individuals with codes G20 (Parkinson’s disease), G21 (secondary parkinsonism), G23 (other degenerative diseases of basal ganglia), G24 (dystonia), or G25 (other extrapyramidal and movement disorders). Neurologically healthy controls were defined as individuals without any neurological or major psychiatric diagnoses. From eligible participants with complete resting-state fMRI data, we selected 208 cases and 208 age- and sex-matched controls.

### **MRI Acquisition and Parcellation**

Resting-state fMRI data were acquired on a Siemens Skyra 3T scanner using a gradient-echo echo-planar imaging sequence with multiband acceleration factor 8, repetition time 0.735 s, echo time 39 ms, flip angle 52°, and 2.4-mm isotropic voxels. Each scan comprised 490 volumes acquired over approximately 6 minutes. Structural T1-weighted images were acquired using a 3D MPRAGE sequence with 1-mm isotropic resolution.

Brain parcellation was performed using a combined cortical–subcortical atlas. The cortical component comprised 360 regions from the HCP-MMP 1.0 atlas, including 180 regions per hemisphere spanning sensory, motor, association, and limbic cortex. The subcortical component comprised 54 regions from the Tian S4 atlas, providing fine-grained subdivision of the thalamus, striatum, pallidum, amygdala, and hippocampus across both hemispheres. The complete 414-region atlas was registered to each subject's native anatomical space using nonlinear transformations computed from the structural scan.

### **Time Series Extraction and Preprocessing**

For each subject, we extracted mean BOLD time series from all 414 regions using the parcellation mask registered to native space. No spatial smoothing, temporal filtering, or global signal regression was applied. We resampled the native 490-timepoint series (total duration 360 s) to 150 timepoints with 2.0-s intervals (total duration 300 s) using cubic spline interpolation. After resampling, we applied z-score normalization independently for each region within each subject, ensuring zero mean and unit variance. This preprocessing order preserves relative temporal structure within subjects while removing amplitude scaling differences.

### **Multi-Cohort Pretraining**

Before fine-tuning on Parkinson’s disease, we pretrained the model sequentially on seven cohorts: (1) epilepsy and status epilepticus (*n* = 847), (2) major depressive episode (*n* = 1,156), (3) mood and affective disorders (*n* = 923), (4) cerebrovascular disease (*n* = 612), (5) substance use disorders (*n* = 734), (6) sleep disorders (*n* = 1,089), and (7) other neurological conditions (*n* = 456). Each cohort was defined using ICD-10 diagnostic codes and matched with neurologically healthy controls. For each cohort, we trained the model for 50 epochs using binary cross-entropy loss before proceeding to the next cohort. Final model weights from each cohort served as initialization for the subsequent cohort.

### **Model Architecture**

The model accepts as input a 150 × 414 matrix representing the time series for all regions. This input is processed through three parallel streams. The primary stream consists of six transformer encoder layers, each with 512-dimensional embeddings, eight query attention heads, and four key-value heads in a grouped-query attention configuration. Rotary position embeddings with dimensionality 64 encode relative temporal positions. Each layer uses root mean square layer normalization before attention and feedforward operations. The feedforward network within each layer employs SwiGLU activation with an expansion factor of 8/3.

The patch embedding stream applies a patch size of two along the temporal dimension, yielding 75 tokens. These patches are projected to 512 dimensions and processed through two transformer layers with the same architecture as the primary stream. Patch representations are integrated with the primary stream through cross-attention in the fourth layer.

The temporal attention pooling module computes attention weights across the 150 timepoints using a two-layer multilayer perceptron with hidden dimension 128 and GELU activation. These weights are applied to the primary stream outputs to produce a single 512-dimensional vector, which is passed through a final linear layer and sigmoid activation for binary classification.

### **Training Procedure**

We partitioned the 416 subjects into training (*n* = 291), validation (*n* = 62), and test (*n* = 63) sets using stratified sampling. After pretraining, we fine-tuned the model on the Parkinson’s disease training set. We used the Ranger21 optimizer with an initial learning rate of 2 × 10⁻⁴, weight decay of 0.05, and cosine annealing schedule. Batch size was set to 32. We monitored validation loss and applied early stopping when loss failed to decrease for 30 consecutive epochs. The final model was selected based on the epoch with lowest validation loss (epoch 217) and evaluated once on the held-out test set.

### **Permutation Feature Importance**

For each of the 414 regions, we computed permutation importance by randomly shuffling the time series of that region across subjects in the validation set and measuring the decrease in F1 score. We repeated this procedure ten times per region, computing the mean and standard deviation of importance scores. Regions were ranked by mean importance score. Statistical significance was assessed by comparing observed importance values to a null distribution generated by shuffling arbitrary non-predictive features.

### **Statistical Analysis**

Classification performance was assessed using accuracy, F1 score, and area under the receiver operating characteristic curve. Confidence intervals were computed using bootstrap resampling with 1,000 iterations. Permutation importance values were compared using nonparametric tests.



## **Data and Code Availability**

UK Biobank data are available to approved researchers through application to UK Biobank. Model code, training scripts, and parcellation atlases will be made publicly available upon publication. Pretrained model weights will be released under an open-source license to facilitate replication and extension to other movement disorders.





### **References**

1. Kish, S. J., Shannak, K., & Hornykiewicz, O. (1988). Uneven pattern of dopamine loss in the striatum of patients with idiopathic Parkinson's disease. *N. Engl. J. Med.*, **318**, 876–880.  
2. Braak, H., et al. (2003). Staging of brain pathology related to sporadic Parkinson's disease. *Neurobiol. Aging*, **24**, 197–211.  
3. Haber, S. N. (2016). The place of dopamine in the cortico-basal ganglia circuit. *Neuroscience*, **333**, 183–197.  
4. Helmich, R. C., et al. (2012). Cerebral causes and consequences of parkinsonian resting tremor. *Brain*, **135**, 3206–3226.  
5. Tinaz, S., et al. (2021). Functional Connectome in Parkinson's Disease. *NeuroImage: Clinical*, **30**, 102675.  
6. Tessitore, A., et al. (2019). Functional Connectivity Signatures of Parkinson's Disease. *NeuroImage*, **185**, 707–716.  
7. Yu, R., et al. (2013). Enhanced Functional Connectivity between Putamen and SMA in PD. *PLoS ONE*, **8**(3), e59717.  
8. Xing, Y., et al. (2020). Coordinate-based meta-analysis of motor functional imaging in PD. *PLoS ONE*, **15**(6), e0234641.  
9. Wright, N., et al. (2020). Elevated caudate connectivity in cognitively normal PD. *Movement Disorders*, **35**(8), 1426–1435.  
10. Khan, S., et al. (2025). Neurostructural changes in Parkinson’s disease. *NeuroImage: Clinical*, **45**, 103712.  
11. Rahimpour, S., et al. (2021). The Supplementary Motor Complex in Parkinson’s Disease. *Front. Neurol.*, **12**, 665413.  
12. Choi, S., et al. (2021). rTMS on SMA for PD: A randomized controlled study. *Brain Stimul.*, **14**(3), 570–578.  
13. Moussawi, K., et al. (2022). DBS effect on anterior pallidum reduces motor impulsivity in PD. *Brain Stimul.*, **15**(1), 123–131.  
14. Manes, J. L., et al. (2018). Altered resting-state FC of putamen and GPi in PD speech impairment. *NeuroImage: Clinical*, **20**, 1161–1169.  
15. Johnson, K. A., et al. (2023). GPi DBS evokes resonant neural activity for targeting. *Ann. Neurol.*, **93**(2), 321–332.  
16. Shine, J. M., et al. (2014). Dysfunctional attentional control networks in PD visual misperceptions. *Brain*, **137**(5), 1513–1523.  
17. Bezdicek, O., et al. (2018). Mild cognitive impairment disrupts attention network connectivity in PD. *NeuroImage: Clinical*, **19**, 543–551.  
18. Su, J., et al. (2021). RoFormer: Enhanced transformer with rotary position embedding. *arXiv:2104.09864*.  
19. Ainslie, J., et al. (2023). GQA: Training generalized multi-query transformer models from multi-head checkpoints. *arXiv:2305.13245*.  
20. Zhang, B., & Sennrich, R. (2019). Root mean square layer normalization. *arXiv:1910.07467*.  
21. Shazeer, N. (2020). GLU variants improve transformer. *arXiv:2002.05202*.

